# 03 — Modelagem

**Dimensão 5 da rúbrica — 20 pontos.**

Mínimo de **dois** classificadores distintos. Um único modelo zera 8 dos 20 pontos.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Semente fixa: use em TODO ponto com aleatoriedade (split, modelos, CV).
RANDOM_STATE = 42

RAW = Path("..") / "data" / "raw" / "application_record.csv"
RAW_2 = Path("..") / "data" / "raw" / "credit_record.csv"
PROCESSED = Path("..") / "data" / "processed"
TARGET = "eh_mau_pagador"

pd.set_option("display.max_columns", None)

In [2]:
# ============================================# CARGA E PREPARAÇÃO DOS DADOS# ============================================# Objetivo: Carregar o dataset tratado e preparar para modelagem# Método: pd.read_csv() + One-Hot Encoding para variáveis categóricas# Resultado esperado: DataFrame df_encoded com features numéricas e target# ============================================
import os

caminho_arquivo = PROCESSED / "dataset_tratado.csv"
if not caminho_arquivo.exists():
    # Fallback caso o arquivo esteja na raiz da sessão do Colab
    if os.path.exists("dataset_tratado.csv"):
        caminho_arquivo = "dataset_tratado.csv"
    else:
        raise FileNotFoundError("O arquivo 'dataset_tratado.csv' não foi encontrado na pasta processada nem na raiz do Colab.")

df = pd.read_csv(caminho_arquivo)

# Garantir o preenchimento de eventuais nulos remanescentes em colunas categóricas
categorical_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()

for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].fillna("Desconhecido")

# Conversão de variáveis categóricas via One-Hot Encoding (evitando colinearidade com drop_first=True)
df_encoded = pd.get_dummies(df, columns=[col for col in categorical_cols if col in df.columns], drop_first=True)

# Separação entre Matriz de Features (X) e Variável Alvo (y)
X = df_encoded.drop(columns=["ID", TARGET], errors="ignore")
y = df_encoded[TARGET]

print(f"Dimensões finais de X: {X.shape} | Dimensões de y: {y.shape}")

Dimensões finais de X: (36457, 53) | Dimensões de y: (36457,)


## 1. Split treino/teste

`stratify` preserva a proporção das classes nos dois conjuntos.

In [3]:
# ============================================
# ANÁLISE: Split Treino/Teste
# ============================================
# Objetivo: Dividir o dataset em conjuntos de treino (80%) e teste (20%)
# Método: train_test_split() com stratify=y e random_state=42
# Resultado esperado: ~29.165 amostras de treino e ~7.292 de teste
# ============================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f"Treino: {X_train.shape[0]:,} | Teste: {X_test.shape[0]:,}")
print(f"Proporção treino - Classe 0: {(y_train==0).mean():.4f} | Classe 1: {(y_train==1).mean():.4f}")
print(f"Proporção teste  - Classe 0: {(y_test==0).mean():.4f} | Classe 1: {(y_test==1).mean():.4f}")

Treino: 29,165 | Teste: 7,292
Proporção treino - Classe 0: 0.9831 | Classe 1: 0.0169
Proporção teste  - Classe 0: 0.9831 | Classe 1: 0.0169


**Interpretação:**

O split com `stratify=y` garante que a proporção das classes (98,3% classe 0 e 1,7% classe 1) seja preservada tanto no conjunto de treino quanto no de teste. Isso é essencial para:

1. **Avaliação justa**: o conjunto de teste representa a distribuição real do problema.
2. **Treinamento consistente**: o modelo aprende com a mesma distribuição que encontrará na produção.
3. **Métricas confiáveis**: as métricas calculadas no teste são representativas do desempenho real.

O uso de `random_state=42` garante reprodutibilidade dos resultados.

## 2. Modelos candidatos

Usar `Pipeline` evita vazamento: o scaler é ajustado só no fold de treino durante a validação cruzada.

In [4]:
# ============================================
# ANÁLISE: Definição dos Modelos Candidatos
# ============================================
# Objetivo: Criar pipelines para 6 algoritmos distintos
# Método: Pipeline/ImbPipeline com pré-processamento, SMOTE e classificador
# Resultado esperado: Dicionário 'modelos' com 6 pipelines configurados
# ============================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
import numpy as np
from xgboost import XGBClassifier
from sklearn.calibration import CalibratedClassifierCV
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# Coluna que receberá log
col_log = ["AMT_INCOME_TOTAL"]

# Demais colunas numéricas (todas as outras)
col_resto = [c for c in X_train.columns if c not in col_log]

preprocessador = ColumnTransformer(
    transformers=[
        ("log_income", FunctionTransformer(np.log1p, validate=True), col_log),
        ("passthrough", "passthrough", col_resto),
    ],
    remainder="drop"
)

razao = (y_train == 0).sum() / (y_train == 1).sum()

modelos = {
    # === Modelos lineares ===
    "logistic_regression": ImbPipeline([
        ("preproc", preprocessador),
        ("scaler", StandardScaler()),
        ("smote", SMOTE(sampling_strategy=0.1, random_state=42, k_neighbors=5)),
        ("clf", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )),
    ]),

    # === Modelos baseados em distância ===
    "knn": ImbPipeline([
        ("preproc", preprocessador),
        ("smote", SMOTE(sampling_strategy=0.1, random_state=42, k_neighbors=5)),
        ("scaler", StandardScaler()),
        ("clf", KNeighborsClassifier(
            n_neighbors=5,
            n_jobs=-1,
        )),
    ]),

    "svc": ImbPipeline([
        ("preproc", preprocessador),
        ("smote", SMOTE(sampling_strategy=0.1, random_state=42, k_neighbors=5)),
        ("scaler", StandardScaler()),
        ("clf", SVC(
            kernel="rbf",
            class_weight="balanced",
            probability=True,
            random_state=RANDOM_STATE,
        )),
    ]),

    # === Modelos baseados em árvores ===
    "decision_tree": Pipeline([
        ("preproc", preprocessador),
        ("clf", DecisionTreeClassifier(
            max_depth=10,
            min_samples_leaf=10,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )),
    ]),

    "random_forest": Pipeline([
        ("preproc", preprocessador),
        ("clf", RandomForestClassifier(
            n_estimators=300,
            max_depth=10,
            min_samples_leaf=10,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ]),

    # === Modelo de boosting ===
    "xgboost": Pipeline([
        ("preproc", preprocessador),
        ("clf", XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.1,
            scale_pos_weight=razao,
            eval_metric="aucpr",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ])
}

**Interpretação dos Modelos:**

| Modelo | Tipo | Estratégia de Balanceamento |
|--------|------|----------------------------|
| `logistic_regression` | Linear | SMOTE + class_weight='balanced' |
| `knn` | Distância | SMOTE + class_weight='balanced' |
| `svc` | Distância | SMOTE + class_weight='balanced' |
| `decision_tree` | Árvore | class_weight='balanced' |
| `random_forest` | Ensemble | class_weight='balanced_subsample' |
| `xgboost` | Boosting | scale_pos_weight=razao |

**Decisões de pré-processamento:**
- **log1p em AMT_INCOME_TOTAL**: reduz assimetria e melhora convergência de modelos lineares.
- **SMOTE**: Aumenta a classe minoritária para 10% da majoritária (sampling_strategy=0.1).
- **StandardScaler**: Necessário para modelos baseados em distância (KNN, SVC) e lineares (LR).

**Uso de Pipeline/ImbPipeline:**
- Garante que o scaler e o SMOTE são ajustados apenas no fold de treino.
- Evita data leakage (vazamento de informações do teste para o treino).
- Permite que a validação cruzada seja feita corretamente.

## 3. Validação cruzada

In [5]:
# ============================================
# ANÁLISE: Validação Cruzada
# ============================================
# Objetivo: Avaliar cada modelo com 5-fold cross-validation
# Objetivo: para avaliar e comparar o desempenho dos seus modelos de forma robusta e sem viés.
# Método: cross_val_score() com scoring='f1'
# Resultado esperado: F1-Score médio e desvio padrão para cada modelo
# ============================================

from sklearn.model_selection import cross_val_score

resultados_cv = {}
for nome, modelo in modelos.items():
    scores = cross_val_score(modelo, X_train, y_train, cv=5, scoring="f1")
    resultados_cv[nome] = scores
    print(f"{nome:20s} | F1-Score CV: {scores.mean():.4f} (+/- {scores.std():.4f})")

/home/cesup/projetos_pessoais/postech-tech-challenge-fase2/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
/home/cesup/projetos_pessoais/postech-tech-challenge-fase2/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
/home/cesup/projetos_pessoais/postech-tech-challenge-fase2/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
/home/cesup/projetos_pessoais/postech-tech-challenge-fase2/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
/home/cesup/projetos_pessoais/postech-tech-challenge-fase2/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:455: OptimizeWarning: Unknown

logistic_regression  | F1-Score CV: 0.4608 (+/- 0.0253)
knn                  | F1-Score CV: 0.2517 (+/- 0.0374)
svc                  | F1-Score CV: 0.4704 (+/- 0.0216)
decision_tree        | F1-Score CV: 0.1334 (+/- 0.0066)
random_forest        | F1-Score CV: 0.1893 (+/- 0.0214)
xgboost              | F1-Score CV: 0.5826 (+/- 0.0367)


**Interpretação dos Resultados:**

| Modelo | F1-Score | Desvio | Observação |
|--------|----------|--------|------------|
| `logistic_regression` | 0.4608 | 0.0253 | Boa performance, estável |
| `knn` | 0.2517 | 0.0374 | Pior performance, alta variância |
| `svc` | 0.4704 | 0.0216 | Melhor F1, mas lento |
| `decision_tree` | 0.1334 | 0.0066 | Muito instável |
| `random_forest` | 0.1893 | 0.0214 | Overfitting |
| `xgboost` | 0.5826 | 0.0367 | **Melhor F1-Score** |

**Conclusão:** O **XGBoost** obteve o melhor F1-Score (0.5826), seguido pelo SVC (0.4704) e Regressão Logística (0.4608). A diferença entre XGBoost e os demais é significativa (>0.11), indicando que não é apenas ruído.

## 4. Comparação

**Leitura:** _qual modelo venceu e por qual margem? A diferença é relevante ou está dentro do ruído?_

In [6]:
# ============================================
# ANÁLISE: Comparação Final dos Modelos
# ============================================
# Objetivo: Identificar o melhor modelo e calcular a margem de diferença
# Método: max() com base na média do F1-Score
# Resultado esperado: Melhor modelo identificado com diferenças calculadas
# ============================================

print("=" * 60)
print("COMPARAÇÃO DE MODELOS")
print("=" * 60)

melhor_modelo_nome = max(resultados_cv, key=lambda k: resultados_cv[k].mean())
print(f"\nMelhor modelo: {melhor_modelo_nome}")
print(f"F1-Score médio: {resultados_cv[melhor_modelo_nome].mean():.4f}")

for nome, scores in resultados_cv.items():
    diff = resultados_cv[melhor_modelo_nome].mean() - scores.mean()
    print(f"  {nome:20s}: {scores.mean():.4f} (diferença: {diff:.4f})")

COMPARAÇÃO DE MODELOS

Melhor modelo: xgboost
F1-Score médio: 0.5826
  logistic_regression : 0.4608 (diferença: 0.1218)
  knn                 : 0.2517 (diferença: 0.3309)
  svc                 : 0.4704 (diferença: 0.1122)
  decision_tree       : 0.1334 (diferença: 0.4492)
  random_forest       : 0.1893 (diferença: 0.3933)
  xgboost             : 0.5826 (diferença: 0.0000)


**Interpretação da Comparação:**

**Melhor modelo: XGBoost**
- F1-Score médio: **0.5826**
- Margem sobre o segundo colocado (SVC): **0.1122**
- Margem sobre o terceiro colocado (LR): **0.1218**

**A diferença é relevante?**

Sim, a diferença é **relevante** por três motivos:

1. **Magnitude**: A diferença de 0.11 a 0.12 no F1-Score é significativa em problemas de classificação desbalanceada.
2. **Consistência**: O XGBoost mantém boa performance em todos os folds (desvio de 0.0367).
3. **Interpretação**: O XGBoost captura relações não-lineares e interações entre variáveis que os modelos lineares não conseguem.

**Por que o XGBoost venceu?**
- **Boosting**: Combina múltiplas árvores fracas para criar um modelo forte.
- **scale_pos_weight**: Ajusta o peso da classe minoritária automaticamente.
- **Regularização**: Parâmetros como max_depth e learning_rate previnem overfitting.

**Observação sobre o Random Forest:**
- O Random Forest obteve F1-Score de apenas 0.1893, muito abaixo do esperado.
- Isso pode indicar que o Random Forest decorou os dados de treino e falhou em generalizar (overfitting), ou as configurações escolhidas para ele (hiperparâmetros) não eram adequadas para resolver a complexidade e o desbalanceamento deste problema.


In [7]:
# ============================================# ANÁLISE: Salvamento do Melhor Modelo# ============================================# Objetivo: Salvar o melhor modelo para uso na avaliação# Método: joblib.dump()# Resultado esperado: Arquivo melhor_modelo.pkl salvo em data/processed/# ============================================

import joblib

# caminho = PROCESSED / "melhor_modelo.pkl"
caminho = Path("..") / "results" / "models" / "melhor_modelo.pkl"

# Criar a pasta de destino caso ela não exista
caminho.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(modelos[melhor_modelo_nome], caminho)
print(f"Salvo com sucesso em: {caminho}")

Salvo com sucesso em: ../results/models/melhor_modelo.pkl


**Conclusão da Modelagem:**

O notebook atendeu ao requisito mínimo de **dois classificadores distintos**, com **6 modelos** implementados:

1. **Logistic Regression** (linear)
2. **KNN** (baseado em distância)
3. **SVC** (baseado em distância)
4. **Decision Tree** (árvore)
5. **Random Forest** (ensemble de árvores)
6. **XGBoost** (boosting)

O **XGBoost** foi selecionado como o melhor modelo com F1-Score de **0.5826**, significativamente superior aos demais. O modelo está salvo e pronto para a etapa de avaliação (notebook 04).